In [1]:
import pandas as pd
import requests
import zipfile
import io

## ETL Prototyping Stocks of Specified Dairy Products
https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3210000101

In [45]:

product_id = "32100001"

url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{product_id}/en"

# Get download URL
response = requests.get(url)
download_url = response.json()["object"]

# Download ZIP
zip_data = requests.get(download_url).content

# Read CSV from ZIP
with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    csv_file = [f for f in z.namelist() if f.endswith(".csv") and "_MetaData" not in f][0]
    df = pd.read_csv(z.open(csv_file))
    df.to_csv("Stocks of specified dairy products", index = False)

C:\Users\asoro\AppData\Local\Temp\ipykernel_14544\1206309277.py:15: DtypeWarning: Columns (0: TERMINATED) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(csv_file))


In [ ]:
df = pd.read_csv("Stocks of specified dairy products")
filtered_df = df[df['Commodity'] == 'Whey powder']

# Most other columns are artifacts from the API call, so we drop them
# Other columns were NULL for all rows, so we drop them as well
filtered_df = filtered_df[['REF_DATE','GEO','Stocks','Commodity','UOM','VALUE']]
# We will keep Stocks for now as it defines two seperate stocks
# Total Stocks - Representing stock of all of canada including imports
# Manufacture and Government stocks - stocks held by domestic manufactures and government stock of dairy

# We will rename the columns to improve readability
filtered_df.rename(columns ={
    "REF_DATE": 'date',
    "GEO": 'country',
    "Stocks": 'stock_type',
    "Commodity": 'commodity',
    "UOM": 'unit',
    "VALUE": 'total_value'
}, inplace = True)




Filtered Values
Number of unique total values: 984
Number of rows: 878
Unfiltered Values
Number of unique total values: 24010
Number of rows: 40540


C:\Users\asoro\AppData\Local\Temp\ipykernel_14544\3327730053.py:1: DtypeWarning: Columns (0: TERMINATED) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Stocks of specified dairy products")


## ETL Prototyping Stocks of Specified dairy products quartlery
https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3210048001
Note that this replaces our previous web source, government switched from monthly to quarterly updates
Script for live updates should only rely on this one, and be called every quarter

In [44]:

product_id = "32100480"

url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{product_id}/en"

# Get download URL
response = requests.get(url)
download_url = response.json()["object"]

# Download ZIP
zip_data = requests.get(download_url).content

# Read CSV from ZIP
with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    csv_file = [f for f in z.namelist() if f.endswith(".csv") and "_MetaData" not in f][0]
    df = pd.read_csv(z.open(csv_file))
    df.to_csv("Stocks of specified dairy products Quarterly", index = False)

In [68]:
df = pd.read_csv("Stocks of specified dairy products Quarterly")

# This dataset has the same issues in different columns
# We can keep the same columns as previous dataset

filtered_df = df[['REF_DATE','GEO','Stocks','Commodity','UOM','VALUE']]

filtered_df.rename(
    columns={
        "REF_DATE": 'date',
        "GEO": 'country',
        "Stocks": 'stock_type',
        "Commodity": 'commodity',
        "UOM": 'unit',
        "VALUE": 'total_value'
    }, inplace=True
)

filtered_df = filtered_df[filtered_df['commodity'] == 'Whey powder']

In [ ]:
filtered_df["total_value"].nunique()
# All Dates are for the firs month of every quarter
# We want it to be monthly updates so
#So we will fill in the gaps for the months that are missing
#Divide the totals evenly across each month - Maybe a later implementation could pull data
# from somewhere else to get a more accurate monthly estimate (Such as demand for whey), 
# but for now this will do

filtered_df.head(10)
# Same total shows for both stock types, investigate to see if we can remove one of the stock types

print(f"Filtered Values")
print(f"Number of unique total values: {filtered_df['total_value'].nunique()*2}")
print(f"Number of rows: {filtered_df.shape[0]}")

print(f"Unfiltered Values")
print(f"Number of unique total values: {df['VALUE'].nunique()*2}")
print(f"Number of rows: {df.shape[0]}")

#It looks like that total stock and manufacture and government stock are the same
#However, in the unfiltered dataset, there are some differences in the total values for the two stock types
#Further investigation is needed to determine if we can remove one of the stock types, or if we need to keep both for some reason

#Manufacture and government stock are still collected under the definition held here
# Inventory Statement of Butter and Cheese - 2026
    # Include:
    # inventory for all dairy products held in your establishment(s), whether owned by you or by others
    # inventory stored in specially rented rooms to which only you have access (except in emergency)
    # stocks held on government accounts.
# Likewise here,
# Dairy Factory Production and Stocks Survey – 2026
    # Report stock values at the end of business on the last day of the quarter.
#  However, Exclude stocks held on Canadian Dairy Commission accounts.



Filtered Values
Number of unique total values: 12
Number of rows: 12
Unfiltered Values
Number of unique total values: 428
Number of rows: 468


In [70]:
keys = ["date", "country", "commodity", "unit"]

total_stock = (
    filtered_df[filtered_df["stock_type"] == "Total stocks"]
    [keys + ["total_value"]]
    .rename(columns={"total_value": "total_stock_value"})
)

manufacturing_stock = (
    filtered_df[
        filtered_df["stock_type"] == "Manufactures and government stocks"
    ]
    [keys + ["total_value"]]
    .rename(columns={"total_value": "manufacturing_stock_value"})
)

different_values = total_stock.merge(
    manufacturing_stock,
    on=keys
)

different_values = different_values[
    different_values["total_stock_value"]
    != different_values["manufacturing_stock_value"]
]

different_values

,date,country,commodity,unit,total_stock_value,manufacturing_stock_value
